# **The *collector* file (collector.py)**

In [ ]:
# Import libraries and modules

import time
import logging
import re
import string
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
from tqdm import tqdm
from langdetect import detect_langs, DetectorFactory, LangDetectException
from collections import deque

In [ ]:
# Set seed for reproducibility in language detection

DetectorFactory.seed = 1

# Import custom modules and configurations for text preprocessing

from config.emoji_filter import remove_emojis
from config.EN_short_text import COMMON_SHORT_EN_WORDS, COMMON_EN_ABBREVIATIONS_ACRONYMS
    ## COMMON_SHORT_EN_WORDS and COMMON_EN_ABBREVIATIONS_ACRONYMS lists are used in the is_english_short_text function to heuristically determine 
    ## if a short text is likely English, even if it fails statistical language detection. 
    ## The presence of common English words or abbreviations/acronyms can indicate that the text is English, especially when the text is too short for reliable language detection.
    ## These lists are not exhaustive. Adding more common words and abbreviations can improve the accuracy of this heuristic.

TOKEN_PATTERN = re.compile(r"\b\w+\b", flags = re.UNICODE)
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

In [ ]:
# Define text preprocessing functions used for language detection and filtering

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = remove_emojis(text)
    #text = text.lower()
        ## language detection is intentionally performed on case-preserved text.
        ## lowercasing can be enabled if stricter normalization is desired. To this end, uncomment this line: text = text.lower()
    text = text.translate(PUNCT_TABLE)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def is_english_short_text(text):
    # Heuristic fallback for short English texts (i.e., below 20 characters) that fail statistical language detection (e.g., insults, slogans, very short comments, acronyms, etc.).

    tokens = text.lower().split()

    if not tokens:
        return False
    # Reject very long tokens (often noise or non-language)
    if not all(token.isascii() for token in tokens):
        return False

    has_common_word = any(token in COMMON_SHORT_EN_WORDS for token in tokens)
    has_abbrev = any(token in COMMON_EN_ABBREVIATIONS_ACRONYMS for token in tokens)

    # Abbreviations alone are not sufficient
    return has_common_word or (has_abbrev and len(tokens) >= 2)

### *First operation: Manage API constraints and quota limits*

In [ ]:
# Custom exception used to inform that the YouTube Data API usage limit has been exceeded

class YouTubeQuotaError(Exception):
    pass

In [ ]:
def execute_with_retries(request, max_retries = 5):
    # Executes a YouTube Data API request with automatic retry logic.
    # max_retries = 5 is arbitrary. It can be adjusted as needed.
    # If YouTube temporarily blocks requests due to rate limits, the function waits and retries automatically.
    # If the daily API quota is exceeded, the program stops immediately.

    retries = 0
    while retries < max_retries:
        try:
            # Attempt to execute the API request
            return request.execute()
        except HttpError as e:
            error_content = e.content.decode('utf-8')

            # Temporary rate limit reached (short-term restriction)
            if 'rateLimitExceeded' in error_content or 'userRateLimitExceeded' in error_content:
                retries += 1
                wait_time = min(2 ** retries, 64)
                    ## exponential backoff strategy to reduce pressure on the API
                logging.warning(f"Rate limit exceeded. Retrying in {wait_time} seconds")
                time.sleep(wait_time)

            # Daily quota fully consumed (hard stop)
            elif 'quotaExceeded' in error_content:
                logging.error("API quota exceeded")
                raise YouTubeQuotaError("API quota exceeded")

            # Any other unexpected API error
            else:
                raise e

    # All retry attempts failed
    raise YouTubeQuotaError(f"Maximum retry attempts reached ({max_retries}). ""Last error was not recoverable.")

### *Second operation: Manage YouTube API HTTP errors*

In [ ]:
def handle_youtube_http_error(error, *, video_id = None, parent_id = None):
    # Centralizes the handling of YouTube Data API errors that are NOT related to rate limits or quotas.
    # Decides whether an error can be ignored safely (for instance, comments disabled) or if it must stop execution.

    error_content = error.content.decode("utf-8", errors = "ignore")

    # Build contextual information for clearer logs
    context = []
        ## create an empty list to store context strings
    if video_id:
        context.append(f"video = {video_id}")
    if parent_id:
        context.append(f"comment = {parent_id}")

    prefix = f"[{' | '.join(context)}] " if context else ""

    # Comments are disabled for this video
    if 'commentsDisabled' in error_content or 'disabled comments' in error_content.lower():
        logging.warning(f"{prefix}Comments are disabled.")
        return "IGNORED"

    # Video does not exist or is private
    if error.resp.status == 404 or 'videoNotFound' in error_content:
        logging.warning(f"{prefix}Video not found or private.")
        return "IGNORED"

    # Access to the resource is forbidden
    if error.resp.status == 403:
        logging.warning(f"{prefix}Access denied (403 error).")
        return "IGNORED"

    # Any other unexpected YouTube Data API error
    logging.error(f"{prefix}Unhandled YouTube API error: {error}")
    raise error

### *Third operation: Data collection*

In [ ]:
def data_metadata_to_retrieve(item, video_id):
    # This function is retrieval-order agnostic. 
    # It concerns only data extraction from API response items at the top-level comment level. Replies are handled separately.

    snippet = item["snippet"]["topLevelComment"]["snippet"]

    return {
        "video_id": video_id,
        "comment_id": item["id"],
        "author": snippet.get("authorDisplayName"),
        "author_id": snippet.get("authorChannelId", {}).get("value", "Not available"),
        "text": snippet.get("textDisplay", ""),
        "like_count": snippet.get("likeCount"),
        "published_at": snippet.get("publishedAt"),
        "updated_at": snippet.get("updatedAt"),
        "is_reply": False}

In [ ]:
def fetch_paginated_results(service, request, max_results = None):
    # Abstracts the pagination logic for YouTube Data API requests.
    # Retrieves all items across pages until there are no more pages or until max_results is reached.

    results = []

    while request:
        response = execute_with_retries(request)
        items = response.get("items", [])
        results.extend(items)

        if max_results and len(results) >= max_results:
            break

        request = service.list_next(request, response)

    return results[:max_results] if max_results else results

In [ ]:
def get_video_comments(youtube, video_id, max_results = None, order = 'relevance', language_filter = True):
    # Retrieves the top-level comments for a single YouTube video.

    comment_threads = youtube.commentThreads()

    request = comment_threads.list(
        part = 'snippet',
        videoId = video_id,
        maxResults = 100,
        textFormat = 'plainText',
        order = "time" if order.lower() == "time_oldest" else order.lower())
            ## 'time_oldest' does not exist for the YouTube Data API (v3). 
            ## Then, this line translates the custom order retrieval into a comprehensible order for the API, as it accepts only "time" or "relevance".

    kept_comments = []
    page_reports = []

    # time_oldest retrieval mode
    if order.lower() == "time_oldest":
        buffer = deque(maxlen = max_results if max_results else None)

        while request:
            response = execute_with_retries(request)
            items = response.get("items", [])
            page_rows = [data_metadata_to_retrieve(i, video_id) for i in items]

            page_df = pd.DataFrame(page_rows)
            if language_filter:
                page_df, report = clean_dataframe(page_df, True)
                page_reports.append(report)

            for row in page_df.to_dict(orient = "records"):
                buffer.append(row)
            
            request = comment_threads.list_next(request, response)
        
        return list(reversed(buffer)), page_reports

    # time and relevance retrieval orders
    while request:
        response = execute_with_retries(request)
        items = response.get("items", [])
        page_rows = [data_metadata_to_retrieve(i, video_id) for i in items]

        page_df = pd.DataFrame(page_rows)
        if language_filter:
            page_df, report = clean_dataframe(page_df, True)
            page_reports.append(report)

        kept_comments.extend(page_df.to_dict(orient = "records"))

        if max_results and len(kept_comments) >= max_results:
            break

        request = comment_threads.list_next(request, response)

    return kept_comments[:max_results] if max_results else kept_comments, page_reports

In [ ]:
def get_replies_to_comments(youtube, parent_comments, max_replies = None):
    # Retrieves replies associated with previously collected top-level comments.

    all_replies = []

    parent_data = [(p.get('comment_id'), p.get('video_id')) for p in parent_comments]

    for parent_id, video_id in parent_data:
        request = youtube.comments().list(
            part = 'snippet',
            parentId = parent_id,
            maxResults = 100,
            textFormat = 'plainText')

        replies_for_parent = []

        while request:
            try:
                response = execute_with_retries(request)
            except HttpError as e:
                if handle_youtube_http_error(e, video_id = video_id, parent_id = parent_id) == "IGNORED":
                    break
                raise

            items = response.get("items", [])

            for item in items:
                snippet = item.get('snippet', {})
                replies_for_parent.append({
                    'video_id': video_id,
                    'comment_id': item.get('id'),
                    'parent_id': parent_id,
                    'author': snippet.get('authorDisplayName'),
                    'author_id': snippet.get('authorChannelId', {}).get('value', 'Not available'),
                    'text': snippet.get('textDisplay', ''),
                    'like_count': snippet.get('likeCount'),
                    'published_at': snippet.get('publishedAt'),
                    'updated_at': snippet.get('updatedAt'),
                    'is_reply': True})

            request = youtube.comments().list_next(request, response)

            if max_replies and len(replies_for_parent) >= max_replies:
                break

        if max_replies and len(replies_for_parent) > max_replies:
            replies_for_parent = replies_for_parent[:max_replies]

        all_replies.extend(replies_for_parent)

    return all_replies

### *Fourth operation: Structure raw data*

In [ ]:
def clean_dataframe(df, language_filter = True):
    ## Produces a dataframe-level report to explain (for instance) why the program needed 
    ## to retrieve more comments than expected to reach the N desired number of comments.
    
    report = {
        "initial_count": len(df),
        "kept_count": 0,
        "excluded_count": 0,
        "because_empty": 0,
        "because_non_english": 0,
        "because_langdetect_failed": 0}

    if df.empty:
        return df, report
      
    # when language filter is disabled
    if not language_filter:
        df["text"] = (df["text"].astype(str).str.replace(r"\r\n|\r|\n", " ", regex = True).str.strip())
        report["kept_count"] = len(df)
        return df.reset_index(drop = True), report    

    # When language filter is enabled
        ## preprocess text (minimal normalization)
    df["text"] = (df["text"].astype(str).str.replace(r"\r\n|\r|\n", " ", regex = True).str.strip())
    
        # remove emoji, punctuation, and whitespace collapse
    df["text"] = df["text"].apply(clean_text)

        # filter out empty comments
    empty_mask = ~df["text"].str.strip().astype(bool)
    report['because_empty'] = int(empty_mask.sum())
    df = df[~empty_mask]

    # language detection and filter out non-English comments
    lang_results = []
    for text in df["text"]:
        try:
            langs = detect_langs(text)
            best = langs[0]
            # Initial check for English via langdetect
            is_en = best.lang == 'en' and best.prob >= (0.70 if len(text) >= 20 else 0.40)

            if not is_en and len(text) < 20:
                is_en = is_english_short_text(text)
                ## use heuristic fallback for short texts if langdetect fails to identify English with sufficient confidence. 
    
            lang_results.append(is_en)
        except LangDetectException:
            if is_english_short_text(text):
                lang_results.append(True)
            else:
                lang_results.append(None)

    lang_series = pd.Series(lang_results, index = df.index)

    report['because_langdetect_failed'] = int(lang_series.isna().sum())
    report['because_non_english'] = int((lang_series == False).sum())

    df = df[lang_series == True]
    
    report['kept_count'] = len(df)
    report['excluded_count'] = (report['because_empty'] + report['because_non_english'] + report['because_langdetect_failed'])

    return df.reset_index(drop = True), report

### *Fifth operation: Orchestration*

In [ ]:
def collect_youtube_comments(api_key, video_ids, max_comments_per_video = None, max_replies_per_comment = None, collect_replies = False, language_filter = True, comment_order = 'relevance'):
    # Main function responsible for collecting top-level comments and replies from multiple YouTube videos.
 
    youtube = build('youtube', 'v3', developerKey = api_key)
    all_data = []
    all_reports = []

    print()
    pbar = tqdm(video_ids, desc = "Video processing status")

    for video_id in pbar:
        try:
            comments, page_reports = get_video_comments(youtube, video_id, max_results = max_comments_per_video, order = comment_order, language_filter = language_filter)

            all_data.extend(comments)
            all_reports.extend(page_reports)

            pbar.set_postfix({"Video ID": video_id, "Comments": len(comments)})

            if collect_replies and comments:
                replies = get_replies_to_comments(youtube, comments, max_replies = max_replies_per_comment)
                all_data.extend(replies)
        
        except HttpError as e:
            logging.error(f"Unexpected error while processing video {video_id}: {e}")
            continue

    data = pd.DataFrame(all_data)

    # Centralized aggregation
    aggregated_report = {
        "initial_count": 0,
        "kept_count": 0,
        "excluded_count": 0,
        "because_empty": 0,
        "because_non_english": 0,
        "because_langdetect_failed": 0,
        "others": 0}        

    for report in all_reports:
        aggregated_report["initial_count"] += report.get("initial_count", 0)
        aggregated_report["because_empty"] += report.get("because_empty", 0)
        aggregated_report["because_non_english"] += report.get("because_non_english", 0)
        aggregated_report["because_langdetect_failed"] += report.get("because_langdetect_failed", 0)

    if language_filter:
        # Only top-level comments count toward N
        aggregated_report["kept_count"] = len(data[data["is_reply"] == False])
        aggregated_report["excluded_count"] = max(0, aggregated_report["initial_count"] - aggregated_report["kept_count"])
        explained_exclusions = (aggregated_report["because_empty"] + aggregated_report["because_non_english"] + aggregated_report["because_langdetect_failed"])
        aggregated_report["others"] = max(0, aggregated_report["excluded_count"] - explained_exclusions)

        filter_report = aggregated_report
    else:
        data, filter_report = clean_dataframe(data, language_filter = False)

    return data, filter_report

# **The *main* file (main.py)**

In [ ]:
# Import libraries and modules, and set up the working environment

import os, sys
import re
import logging

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

In [ ]:
# Import the orchestration function from collector.py and the custom error class raised when the API quota is exceeded
# Import common English words and abbreviations/acronyms lists used for heuristic short text language detection

from collector import collect_youtube_comments, YouTubeQuotaError
from config.EN_short_text import COMMON_SHORT_EN_WORDS, COMMON_EN_ABBREVIATIONS_ACRONYMS

In [ ]:
# Configure how log information and warnings are displayed in the console (time, severity level, message)

logging.basicConfig(level = logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s")

In [ ]:
# Load the API key from the "api_key.txt" file

def load_api_key(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"API key file not found: {filepath}")
    
    with open(filepath, "r") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if line.startswith("YOUTUBE_API_KEY"):
                return line.split("=", 1)[1].strip()
    
    raise ValueError("Missing YouTube API key. Please check the api_key.txt file in the config directory.")

API_KEY = load_api_key("config/api_key.txt")

In [ ]:
# Define a mapping for comment retrieval order options

ORDER_MAP = {"r": "Relevance", "t": "Time", "o": "Time_Oldest"}

### *First operation: Define the unit of analysis*

In [ ]:
def read_video_ids(filepath):
    # Reads YouTube video IDs from a text file.

    # Check if the file exists before reading it
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    # Regular expression defining the official YouTube video ID format
    YOUTUBE_ID_PATTERN = re.compile(r'^[a-zA-Z0-9_-]{11}$')
        ## until now, YouTube video IDs have always been 11 characters long. Please note that this may change in the future and require code updates.
    
    # Create an empty list to store valid video IDs and a counter for invalid IDs
    valid_ids = []
    invalid_ids_count = 0
    
    # Open and read the file sequentially
    with open(filepath, "r") as file:
        for line in file:
            # ignore anything after a '#' character (annotations)
            raw_id = line.split("#")[0].strip()
            
            # skip empty lines
            if not raw_id:
                continue
            
            # check if the ID matches the expected YouTube format
            if YOUTUBE_ID_PATTERN.match(raw_id):
                valid_ids.append(raw_id)
            else:
                logging.warning(f"ID ignored (invalid format): '{raw_id}'. Must be an 11-character ID.")
                ## increment the counter of invalid IDs when an invalid ID is found
                invalid_ids_count += 1
    
    # Stop execution if no valid IDs were found
    if not valid_ids:
        raise ValueError(f"No valid video ID found in {filepath}")

    # Warn the user if some IDs were invalid
    if invalid_ids_count > 0:
        logging.warning(f"\nWARNING: {invalid_ids_count} invalid ID(s) were ignored before data collection.")
        
    return valid_ids

### *Second operation: Configuration of the data collection parameters*

In [ ]:
def get_user_input():
    # Interactively asks the user for all configuration options needed before starting the data collection process.
    
    # Ask how many top-level comments should be collected per video
    max_retries = 3
    for attempt in range(max_retries):
        ## 'attempt' does not need to be incremented manually, as the for loop handles it. It is automatically defined and updated by the loop.
        try:
            max_comments = int(input("How many top-level comments do you want to retrieve per video? (maximum 2000): ").strip())
            ## '(maximum 2000)' is an indication to the user, not a hard limit imposed by the API. It can be adjusted as needed or removed.
            if 1 <= max_comments <= 2000:
                break
            else:
                print("\nEnter a number between 1 and 2000.")
        except ValueError:
            print("\nEnter a valid integer.")
    else:
        # default value if the user fails too many times
        max_comments = 100
        print("\nDefault value of 100 used for top-level comments.")

    # Ask to select the comment retrieval order
    while True:
        comment_order_input = input("\nSelect comment order by pressing the associated key:\n" 
                                    "\n'r' -> Relevance. Please note that the API may return an order that differs from the one displayed on the platform.\n"
                                    "\n't' -> Time. Retrieve top-level comments from newest to oldest.\n"
                                    "\n'o' -> Time_Oldest. Retrieve top-level comments from oldest to newest. Requires full pagination, which may consume a lot of quota on videos with many comments.\n"
                                    "\n > ").strip().lower()

        if comment_order_input in ORDER_MAP:
            comment_order = ORDER_MAP[comment_order_input]
            break
        else:
            print("\nInvalid input. Please try again.")
      
    # Ask whether replies to comments should be collected
    while True:
        collect_replies_user_input = input("\nDo you want to collect comment replies? (yes/no): ").strip().lower()

        if collect_replies_user_input in {'yes', 'no'}:
            collect_replies = collect_replies_user_input == 'yes'
            break
        else:
            print("\nPlease answer 'yes' or 'no'.") 

    # Ask how many replies should be collected per comment (only if enabled)
    max_replies = 0
    if collect_replies:
        for attempt in range(max_retries):
            try:
                max_replies = int(input("\nHow many replies do you want to retrieve per comment? (maximum 50): ").strip())
                ## '(maximum 50)' is an indication to the user, not a hard limit imposed by the API. It can be adjusted as needed or removed.
                if 1 <= max_replies <= 50:
                    break
                else:
                    print("\nEnter a number between 1 and 50.")
            except ValueError:
                print("\nEnter a valid integer.")
        else:
            max_replies = 10
            print("\nDefault value of 10 used for comment replies.")
    
    # Ask whether the language-filtering pipeline should be applied to raw text
    while True:
        language_filter_input = input("\nDo you want to apply the language-filtering pipeline on raw text (i.e., remove emojis, normalize punctuation, and collect only English-written comments)? (yes/no): ").strip().lower()

        if language_filter_input in {'yes', 'no'}:
            language_filter = language_filter_input == 'yes'
            break
        else:
            print("\nPlease answer 'yes' or 'no'.")
    
    # Ask for the output file name
    while True:
        output_filename = input("\nOutput file name (file_name.csv): ").strip()

        # automatically add the .csv extension if missing
        if not output_filename.endswith(".csv"):
            output_filename += ".csv"

        # ensure the file name only contains safe characters
        if re.match(r'^[\w,\s-]+\.csv$', output_filename):
            break
        else:
            print("Invalid file name.")
    
    return collect_replies, max_comments, max_replies, output_filename, language_filter, comment_order

### *Third operation: Execute data collection*

In [ ]:
if __name__ == "__main__":

    print("\n\n-- YouTube Data Collector - Configuration --\n")
    
    # File containing the list of video IDs to process
    video_ids_file = "source/video_ids.txt"

    try:
        video_ids = read_video_ids(video_ids_file)
        print(f"{len(video_ids)} valid video(s) found in {video_ids_file}\n")
    except (FileNotFoundError, ValueError) as e:
        # stop the program if the video IDs cannot be loaded
        logging.error(str(e))
        raise
    
    # Ask the user for all required parameters
    collect_replies, max_comments, max_replies, output_filename, language_filter, comment_order = get_user_input()
    
    print("\n\n-- Start of data collection --\n")
    
    try:
        # Launch the data collection process
        df_comments, language_filter_report = collect_youtube_comments(
            api_key = API_KEY,
            video_ids = video_ids,
            max_comments_per_video = max_comments,
            max_replies_per_comment = max_replies,
            collect_replies = collect_replies,
            language_filter = language_filter,
            comment_order = comment_order)

        # Display a summary if data was collected
        if not df_comments.empty:
            total_comments = len(df_comments[df_comments['is_reply'] == False])
            total_replies = len(df_comments[df_comments['is_reply'] == True])
            
            print(f"\n{'_' * 45}\n"
                "\nDATA COLLECTION SUMMARY:\n"
                f"\n  - Comment order selected -> {comment_order}\n"
                f"\n  - Total top-level comments -> {total_comments}\n"
                f"\n  - Total replies -> {total_replies}\n"
                f"{'_' * 45}\n")
        else:
            print("\nNo data retrieved. Check per-video logs or API status (e.g., commentsDisabled).\n")

        # Display the language filtering summary if the filter was applied and if data were collected (i.e., not empty dataframe)
        if language_filter and not df_comments.empty:
            print(
                f"LANGUAGE FILTERING SUMMARY:\n\n"
                f"Initial number of retrieved comments -> {language_filter_report['initial_count']}\n"
                f"\n  - Comments kept after filtering -> {language_filter_report['kept_count']}\n"
                f"\n  - Comments excluded -> {language_filter_report['excluded_count']}\n"
                f"\nReasons for exclusion:\n"
                f"\n - Empty after preprocessing (emojis and white-space) -> {language_filter_report['because_empty']}\n"
                f"\n - Non-English content detected (too likely being another language than EN) -> {language_filter_report['because_non_english']}\n"
                f"\n - Language detection failed (text too short or ambiguous to be detected) -> {language_filter_report['because_langdetect_failed']}\n"
                f"\n - Others (e.g., API error, max_results reached, exception raised) -> {language_filter_report['others']}\n"
                f"{'_' * 45}\n")

        # Create the output directory if it does not exist
        if not df_comments.empty:
            output_dir = "data"
            os.makedirs(output_dir, exist_ok = True)

            # Build the full path of the output file
            output_path = os.path.join(output_dir, output_filename)
        
            # Save the collected data into a CSV file
            df_comments.to_csv(output_path, index = False, sep = ";", encoding = "utf-8-sig")
        
            print("-- Data collection process completed --\n")
        else:          
            ## remove any pre-existing empty output file
            output_path = os.path.join("data", output_filename)
            try:
                if os.path.exists(output_path) and os.path.getsize(output_path) == 0:
                    os.remove(output_path)
            except Exception:
                pass
        
    except YouTubeQuotaError as e:
        # Stop immediately if the YouTube Data API quota is exceeded
        logging.critical(f"CRITICAL STOP. {str(e)}")
    
    except Exception as e:
        # Catch any unexpected fatal error
        logging.error(f"Unexpected fatal error during data collection: {e}", exc_info = True)